In [0]:
%sql
USE CATALOG fflogs_graphql;

USE SCHEMA gold;

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import DataFrame

In [0]:
%sql
SELECT
    class,
    rDPS,
    aDPS,
    duration
FROM fflogs_encounter_ranking_gold
WHERE encounter_name = 'Vamp Fatale'
;

In [0]:
def get_xy_arr(
    data: DataFrame,
    x_arr_str: str,
    y_arr_str: str,
    sample_size: int = 100
) -> tuple[np.ndarray, np.ndarray]:
    pd_data = (data
        .toPandas()
        .sample(sample_size, replace=True, random_state=42)
    )
    x_arr = pd_data[x_arr_str].to_numpy()
    y_arr = pd_data[y_arr_str].to_numpy()

    return (x_arr, y_arr)

In [0]:
def calc_least_square_line(
    x_arr: np.ndarray,
    y_arr: np.ndarray,
    round_decimal: int | None = None
) -> tuple[float, float]:
    x_ = np.mean(x_arr)
    y_ = np.mean(y_arr)
    xy_ = np.mean(x_arr * y_arr)
    x_sqr_ = np.mean(x_arr ** 2)

    m = (xy_ - (x_ * y_)) / (x_sqr_ - (x_**2))
    b = y_ - (m * x_)

    if round_decimal:
        m = np.round(m, round_decimal)
        b = np.round(b, round_decimal)

    print(f"Line of best fit formula: y = {m:.2f}x + {b:.2f}")

    return (m, b)

In [0]:
def calc_r_squared(
    m: float,
    b: float,
    x_arr: np.ndarray,
    y_arr: np.ndarray,
) -> float:
    # ----- Base Sum of Square Errors
    y_ = np.mean(y_arr)
    base_sse = np.sum((y_arr - y_) ** 2)

    # ----- Linear Regression Sum of Square Errors
    y_pred = m*x_arr + b
    reg_sse = np.sum((y_arr - y_pred) ** 2)

    # ----- Calculate R-Square
    r_sqr = np.round(
        1 - (reg_sse / base_sse)
    , 2)

    print(f"Pre-regression SSE --> {base_sse}")
    print(f"Post-regression SSE --> {reg_sse}")
    print(
        f"The least square line result in an r_sqr result of {r_sqr:.2f}, indicating a {r_sqr * 100}% "
          "reduction in variance error accounted by the x variable."
    )
    return r_sqr

In [0]:
def visualize_scatterplot(
    x_arr: np.ndarray,
    y_arr: np.ndarray,
    m: float | None = None,
    b: float | None = None,
) -> None:
    # ----- Base scatterplot
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.scatterplot(
        x=x_arr,
        y=y_arr,
        ax=ax
    )
    # ----- Overlay line of least square
    if m is not None and b is not None:
        x_vals = np.linspace(x_arr.min(), x_arr.max(), 100)
        y_vals = m * x_vals + b
        
        ax.plot(x_vals, y_vals, color='red', label=f'y = {m:.2f}x + {b:.2f}')
        ax.legend()

    # ----- Decorate and display
    ax.set_title(f"Bivariate relationship + Manual Linear Regression Line")

    plt.show()
    plt.close(fig)

In [0]:
def visualize_regplot(
    x_arr: np.ndarray,
    y_arr: np.ndarray,
) -> None:
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.regplot(
        x=x_arr,
        y=y_arr,
        ax=ax,
    )

    # Extract m and b from the line regplot drew
    line = ax.lines[0]
    x_vals = line.get_xdata()
    y_vals = line.get_ydata()
    
    m = (y_vals[-1] - y_vals[0]) / (x_vals[-1] - x_vals[0])
    b = y_vals[0] - m * x_vals[0]

    ax.legend([f'y = {m:.2f}x + {b:.2f}'])
    ax.set_title(f"Bivariate relationship + Automatic Linear Regression Line")
    plt.show()
    plt.close(fig)

In [0]:
def run() -> None:
    x_arr, y_arr = get_xy_arr(_sqldf, 'rDPS', 'aDPS')
    m, b = calc_least_square_line(x_arr, y_arr, round_decimal=2)
    r_sqr = calc_r_squared(m, b, x_arr, y_arr)

    visualize_scatterplot(
        x_arr=x_arr,
        y_arr=y_arr,
        m=m,
        b=b
    )

    visualize_regplot(x_arr, y_arr)

In [0]:
run()